- Overviews
1. Introduction
3. Data Understanding
4. Data Cleaning
5. Data transformation
6. Simpan data ke MongoDB
7. Kesimpulan

### 1. Introduction

1.1 Peran Penting Data Preparation

1.2 Overview Dataset

In [4]:
%pip install mysql-connector-python

  Using cached mysql_connector_python-9.6.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
    --------------------------------------- 0.3/16.5 MB ? eta -:--:--
    --------------------------------------- 0.3/16.5 MB ? eta -:--:--
    --------------------------------------- 0.3/16.5 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.5 MB 526.9 kB/s eta 0:00:31
   - -------------------------------------- 0.5/16.5 MB 526.9 kB/s eta 0:00:31
   - -------------------------------------- 0.5/16.5 MB 526.9 kB/s eta 0:00:31
   - -------------------------------------- 0.5/16.5 MB 526.9 kB/s eta 0:00:31
   - -------------------------------------- 0.5/16.5 MB 526.9 kB/s eta 0:00:31
   - -------------------------------------- 0.5/16.5 MB 526.9 kB/s eta 0:00:31
   - -------------------------------------- 0.5/16.5 MB 526.9 kB/s eta 0:00:31
   - -----


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 2. Data Understanding

2.1 Import Library

In [8]:
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",      # Laragon default kosong
    database="db_astera"
)

2.2 Read Dataset

In [9]:
query = "SELECT * FROM data_penjualan"
data = pd.read_sql(query, conn)
data.head()

C:\Users\X13\AppData\Local\Temp\ipykernel_2608\1728757909.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data = pd.read_sql(query, conn)


,id,tanggal,total_penjualan,total_pesanan,penjualan_perpesanan,created_at,updated_at
0,606,2020-10-01,615000,7,87857,2026-02-10 20:38:10,2026-02-10 20:38:10
1,607,2020-10-02,340200,4,85050,2026-02-10 20:38:10,2026-02-10 20:38:10
2,608,2020-10-03,0,0,0,2026-02-10 20:38:10,2026-02-10 20:38:10
3,609,2020-10-04,487500,6,81250,2026-02-10 20:38:10,2026-02-10 20:38:10
4,610,2020-10-05,295000,3,98333,2026-02-10 20:38:10,2026-02-10 20:38:10


### 3. Data Cleaning

3.1 Analysis Data

In [10]:
# Jumlah baris dan kolom
print("Jumlah baris dan kolom:")
print(data.shape)

# Tipe data tiap kolom
print("\nTipe data per kolom:")
print(data.dtypes)

# Statistik deskriptif
print("\nStatistik deskriptif:")
print(data.describe(include='all'))

# Cek data duplikat
print("\nJumlah data duplikat:")
print(data.duplicated().sum())

Jumlah baris dan kolom:
(2035, 7)

Tipe data per kolom:
id                               int64
tanggal                         object
total_penjualan                 object
total_pesanan                    int64
penjualan_perpesanan            object
created_at              datetime64[ns]
updated_at              datetime64[ns]
dtype: object

Statistik deskriptif:
                 id     tanggal total_penjualan  total_pesanan  \
count   2035.000000        2035            2035    2035.000000   
unique          NaN        1915            1659            NaN   
top             NaN  2025-01-01               0            NaN   
freq            NaN           2             223            NaN   
mean    1623.000000         NaN             NaN      11.134644   
min      606.000000         NaN             NaN       0.000000   
25%     1114.500000         NaN             NaN       3.000000   
50%     1623.000000         NaN             NaN       7.000000   
75%     2131.500000         NaN         

In [11]:
# Cek missing values per kolom
print("\nJumlah missing values per kolom:")
print(data.isnull().sum())


Jumlah missing values per kolom:
id                      0
tanggal                 0
total_penjualan         0
total_pesanan           0
penjualan_perpesanan    0
created_at              0
updated_at              0
dtype: int64


3.3. Tangani Missing Values

In [15]:


# ================================
# 1. Konversi tipe data
# ================================
data['tanggal'] = pd.to_datetime(data['tanggal'], errors='coerce')

data['total_penjualan'] = pd.to_numeric(
    data['total_penjualan'], errors='coerce'
)

data['penjualan_perpesanan'] = pd.to_numeric(
    data['penjualan_perpesanan'], errors='coerce'
)

# ================================
# 2. Tangani missing values (TANPA inplace)
# ================================
data['tanggal'] = data['tanggal'].fillna(
    data['tanggal'].mode()[0]
)

data['total_penjualan'] = data['total_penjualan'].fillna(
    data['total_penjualan'].median()
)

data['penjualan_perpesanan'] = data['penjualan_perpesanan'].fillna(
    data['penjualan_perpesanan'].median()
)

# ================================
# 3. Drop data penting yang kosong
# ================================
data = data.dropna(subset=['total_pesanan'])

# ================================
# 4. Cek hasil akhir
# ================================
print("Tipe data setelah dibersihkan:")
print(data.dtypes)

print("\nJumlah missing values setelah dibersihkan:")
print(data.isnull().sum())

Tipe data setelah dibersihkan:
id                               int64
tanggal                 datetime64[ns]
total_penjualan                  int64
total_pesanan                    int64
penjualan_perpesanan             int64
created_at              datetime64[ns]
updated_at              datetime64[ns]
dtype: object

Jumlah missing values setelah dibersihkan:
id                      0
tanggal                 0
total_penjualan         0
total_pesanan           0
penjualan_perpesanan    0
created_at              0
updated_at              0
dtype: int64



Missing values:
Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


3.4 Tangani Duplikasi (Jika Ada)

In [18]:
data = data.drop_duplicates(
    subset=['tanggal', 'total_penjualan', 'total_pesanan']
)
# Cek duplikasi
print("\nJumlah duplikat:", data.duplicated().sum())


Jumlah duplikat: 0


### 4. Data Transformation

4.1 Encoding Kolom Kategorikal

In [20]:
data = data.sort_values('tanggal').reset_index(drop=True)
data['hari_ke'] = (data['tanggal'] - data['tanggal'].min()).dt.days + 1
data[['tanggal', 'hari_ke']].head()

,tanggal,hari_ke
0,2020-10-01,1
1,2020-10-02,2
2,2020-10-03,3
3,2020-10-04,4
4,2020-10-05,5


In [21]:
# Variabel X dan y
X = data[['hari_ke']]
y = data['total_penjualan']

print("Contoh X (hari_ke):")
print(X.head())

print("\nContoh y (total_penjualan):")
print(y.head())

Contoh X (hari_ke):
   hari_ke
0        1
1        2
2        3
3        4
4        5

Contoh y (total_penjualan):
0    615000
1    340200
2         0
3    487500
4    295000
Name: total_penjualan, dtype: int64


In [22]:
from sklearn.preprocessing import PolynomialFeatures

# Transformasi polynomial degree 2
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)

print("\nBentuk X setelah transformasi polinomial:")
print(X_poly[:5])
print("Jumlah fitur setelah polinomial:", X_poly.shape[1])


Bentuk X setelah transformasi polinomial:
[[ 1.  1.]
 [ 2.  4.]
 [ 3.  9.]
 [ 4. 16.]
 [ 5. 25.]]
Jumlah fitur setelah polinomial: 2


In [23]:
# Data training (2020–2024)
X_train = X_poly[data['tahun'] <= 2024]
y_train = y[data['tahun'] <= 2024]

# Data testing (2025)
X_test = X_poly[data['tahun'] == 2025]
y_test = y[data['tahun'] == 2025]

print("\nJumlah data training:", X_train.shape[0])
print("Jumlah data testing:", X_test.shape[0])


Jumlah data training: 1550
Jumlah data testing: 365


In [24]:
print("\nContoh X_train:")
print(X_train[:5])

print("\nContoh y_train:")
print(y_train.head())


Contoh X_train:
[[ 1.  1.]
 [ 2.  4.]
 [ 3.  9.]
 [ 4. 16.]
 [ 5. 25.]]

Contoh y_train:
0    615000
1    340200
2         0
3    487500
4    295000
Name: total_penjualan, dtype: int64


### 5 Simpan Data to MongoDB

In [25]:
cursor.executemany(insert_query, data_sql)
conn.commit()

print("Data berhasil disimpan ke MySQL")

NameError: name 'cursor' is not defined

In [144]:
print("Jumlah dokumen dalam koleksi:", collection.count_documents({}))

Jumlah dokumen dalam koleksi: 773


### 6. Kesimpulan

Yang sudah dilakukan:
- Belajar pentingnya data preparation.

- Memahami data Titanic: struktur, tipe, missing values, dll.

- Membersihkan data: tangani missing, duplikat, dll.

- Transformasi data: encoding, normalisasi, dll.

Berikutnya:
- Melakukan visualisasi data eksploratif untuk memahami hubungan antar fitur dan target.